## Initial preprocessing of an EEG file

In [10]:
# Installing and Importing MNE
%pip install mne
import mne
import os
import glob

%pip install mne-qt-browser PyQt5
mne.viz.set_browser_backend('qt')

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


'qt'

In [11]:
# Reading a .gdf file from the dataset
filename = 'data/ID1.gdf' # Change here to run another file
raw = mne.io.read_raw_gdf(filename, preload=True)

Extracting EDF parameters from c:\Users\Renan\Pictures\Universidade\8periodo\wireless-eeg-cnp-analysis\data\ID1.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, P8, Fz, Cz, Pz, M1, M2, AFz, CPz, POz
Creating raw.info structure...
Reading 0 ... 151455  =      0.000 ...   605.820 secs...


In [12]:
# raw data
print(raw.info)

raw.plot(title='Raw Data')

<Info | 8 non-empty values
 bads: []
 ch_names: FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, ...
 chs: 24 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 125.0 Hz
 meas_date: unspecified
 nchan: 24
 projs: []
 sfreq: 250.0 Hz
 subject_info: <subject_info | his_id: X, last_name: >
>


Channels marked as bad:
none


In [13]:
# Define the input and output folders
input_folder = 'data'
output_folder = 'cleaned'

# Create the 'cleaned' folder if it doesn't exist yet
os.makedirs(output_folder, exist_ok=True)

# Find all .gdf files in the input folder
# glob will return a list with the paths, e.g., ['data/ID1.gdf', 'data/ID2.gdf', ...]
gdf_files = glob.glob(os.path.join(input_folder, '*.gdf'))

# Check if files were found
if not gdf_files:
    print(f"No .gdf files found in the '{input_folder}' folder.")
else:
    print(f"Found {len(gdf_files)} files to process.\n")

# Loop to process each file
for file in gdf_files:
    # Extract only the file name without the extension and folder (e.g., 'ID1')
    base_name = os.path.splitext(os.path.basename(file))[0]
    
    # Read the .gdf file
    raw = mne.io.read_raw_gdf(file, preload=True)
    raw_proc = raw.copy()
    
    # Set the electrode montage
    montage = mne.channels.make_standard_montage('standard_1020')
    raw_proc.set_montage(montage, match_case=False, on_missing='warn')
    
    # Frequency Filtering (Band-pass filter from 1.0 to 40.0 Hz)
    raw_proc.filter(l_freq=1.0, h_freq=40.0)
    
    # Notch Filter (remove power line noise - 60 Hz)
    power_line_freq = 60.0 
    raw_proc.notch_filter(freqs=power_line_freq)
    
    # Re-referencing
    raw_proc.set_eeg_reference('average', projection=True)
    raw_proc.apply_proj()
    
    # Configure ICA (using 15 components, since we have 24 channels)
    ica = mne.preprocessing.ICA(n_components=15, random_state=97, max_iter='auto')
    
    # Fit ICA on the already filtered data
    ica.fit(raw_proc)
    
    # Automatically find the components related to eye blinks
    # We use the 'FP1' channel as a reference to capture eye movement
    bads_eog, scores_eog = ica.find_bads_eog(raw_proc, ch_name='FP1')
    
    # Mark these components as "bad" to be removed
    ica.exclude = bads_eog
    print(f"Components automatically removed by ICA: {bads_eog}")
    
    # Apply ICA to clean the signal (this permanently alters raw_proc)
    ica.apply(raw_proc)

    # Define the path where the file will be dynamically saved
    save_filename = f"{base_name}_preproc_eeg.fif"
    save_path = os.path.join(output_folder, save_filename)
    
    # Save the processed data
    raw_proc.save(save_path, overwrite=True)

print("Processing of all files completed!")

Found 37 files to process.

Extracting EDF parameters from c:\Users\Renan\Pictures\Universidade\8periodo\wireless-eeg-cnp-analysis\data\ID0.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, P8, Fz, Cz, Pz, M1, M2, AFz, CPz, POz
Creating raw.info structure...
Reading 0 ... 149951  =      0.000 ...   599.804 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filte

In [14]:
# Reading a .fif file from the cleaned folder
preproc_file = 'cleaned/ID1_preproc_eeg.fif' # Change here to run another file
raw_preproc = mne.io.read_raw_fif(preproc_file, preload=True)

Opening raw data file cleaned/ID1_preproc_eeg.fif...
    Read a total of 1 projection items:
        Average EEG reference (1 x 24) active
    Range : 0 ... 151455 =      0.000 ...   605.820 secs
Ready.
Reading 0 ... 151455  =      0.000 ...   605.820 secs...


In [15]:
# raw_preproc data
print(raw_preproc.info)

raw_preproc.plot(title='Preprocessed Data')

<Info | 12 non-empty values
 bads: []
 ch_names: FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, ...
 chs: 24 EEG
 custom_ref_applied: False
 dig: 27 items (3 Cardinal, 24 EEG)
 file_id: 4 items (dict)
 highpass: 1.0 Hz
 lowpass: 40.0 Hz
 meas_date: unspecified
 meas_id: 4 items (dict)
 nchan: 24
 projs: Average EEG reference: on
 sfreq: 250.0 Hz
 subject_info: <subject_info | his_id: X>
>


Channels marked as bad:
none
